In [ ]:
import pandas as pd
from multicall import Call


from mainnet_launch.database.schema.full import (
    DestinationTokenValues,
    DestinationTokens,
    Destinations,
    Tokens,
    TokenValues,
    Autopools,
)


from mainnet_launch.database.schema.postgres_operations import (
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
    get_full_table_as_orm,
    TableSelector,
    merge_tables_as_df,
)

from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
)

from mainnet_launch.constants import ALL_CHAINS, ROOT_PRICE_ORACLE, ChainData


def _fetch_destination_token_value_data_from_external_source(
    chain: ChainData, possible_blocks: list[int], full_destination_df: pd.DataFrame
) -> pd.DataFrame:

    def build_pool_token_spot_price_calls(
        chain: ChainData, pool_addresses: list[str], token_addresses: list[str]
    ) -> list[Call]:
        return [
            Call(
                ROOT_PRICE_ORACLE(chain),
                ["getSpotPriceInEth(address,address)(uint256)", token_address, pool_address],
                [((pool_address, token_address, "spot_price"), safe_normalize_with_bool_success)],
            )
            for (pool_address, token_address) in zip(pool_addresses, token_addresses)
        ]

    def build_underlying_reserves_calls(destinations: list[str]) -> list[Call]:
        return [
            Call(
                dest,
                "underlyingReserves()(address[],uint256[])",
                [
                    ((dest, "underlyingReserves_tokens"), identity_with_bool_success),
                    ((dest, "underlyingReserves_amounts"), identity_with_bool_success),
                ],
            )
            for dest in destinations
        ]

    spot_price_calls = build_pool_token_spot_price_calls(
        chain, full_destination_df["pool"], full_destination_df["token_address"]
    )

    underlying_reserves_calls = build_underlying_reserves_calls(full_destination_df["destination_vault_address"])

    return get_raw_state_by_blocks(
        [*spot_price_calls, *underlying_reserves_calls], possible_blocks, chain, include_block_number=True
    )


def _build_all_destination_token_values(
    chain: ChainData, full_destination_df: pd.DataFrame, wide_df: pd.DataFrame
) -> list[DestinationTokenValues]:
    all_destination_token_values = []

    def _build_destination_token_values(row: dict):
        # I suspect this is causing an issue
        amounts = row["underlyingReserves_amounts"]
        index = row["index"]
        decimals = row["decimals"]

        if amounts is not None:
            quantity = amounts[index] / (10**decimals)
        else:
            quantity = None
            pass

        all_destination_token_values.append(
            DestinationTokenValues(
                block=row["block"],
                chain_id=chain.chain_id,
                destination_vault_address=row["destination_vault_address"],
                token_address=row["token_address"],
                spot_price=row["spot_price"],
                quantity=quantity,
            )
        )

    for destination_vault_address, pool_address, token_address, index in zip(
        full_destination_df["destination_vault_address"],
        full_destination_df["pool"],
        full_destination_df["token_address"],
        full_destination_df["index"],
    ):
        cols = [
            (pool_address, token_address, "spot_price"),
            (destination_vault_address, "underlyingReserves_amounts"),
            "block",
        ]

        this_destination_token_df = wide_df[cols].copy()
        this_destination_token_df.columns = ["spot_price", "underlyingReserves_amounts", "block"]
        this_destination_token_df["token_address"] = token_address
        this_destination_token_df["index"] = index
        this_destination_token_df["destination_vault_address"] = destination_vault_address

        this_destination_token_df.apply(_build_destination_token_values, axis=1)

    return all_destination_token_values


def ensure_destination_token_values_are_current():
    for chain in ALL_CHAINS:
        possible_blocks = build_blocks_to_use(chain)

        missing_blocks = get_subset_not_already_in_column(
            DestinationTokenValues,
            DestinationTokenValues.block,
            possible_blocks,
            where_clause=DestinationTokenValues.chain_id == chain.chain_id,
        )
        missing_blocks = possible_blocks[::30]
        if len(missing_blocks) == 0:
            continue

        destinations_df = merge_tables_as_df(
            [
                TableSelector(DestinationTokens, [DestinationTokens.token_address, DestinationTokens.index]),
                TableSelector(
                    Destinations,
                    [DestinationTokens.destination_vault_address],
                    join_on=(Destinations.chain_id == DestinationTokens.chain_id)
                    & (Destinations.destination_vault_address == DestinationTokens.destination_vault_address),
                ),
                TableSelector(
                    Tokens,
                    [Tokens.decimals],
                    join_on=(Tokens.chain_id == DestinationTokens.chain_id)
                    & (Tokens.token_address == DestinationTokens.token_address),
                ),
            ],
            where_clause=(DestinationTokens.chain_id == chain.chain_id) & (Destinations.pool_type != "idle"),
        )

        token_spot_prices_and_reserves_df = _fetch_destination_token_value_data_from_external_source(
            chain, missing_blocks, destinations_df
        )

        new_destination_token_values_rows = []

        def _extract_destination_token_values(row: dict) -> None:
            token_spot_price_column = (row["pool"], row["token_address"], "spot_price")
            quantity_column = (row["destination_vault_address"], "underlyingReserves_amounts")

            sub_df = token_spot_prices_and_reserves_df[["block", quantity_column, token_spot_price_column]].copy()
            sub_df.columns = ["block", "quantity", "spot_price"]
            sub_df["quantity"] = sub_df["quantity"].apply(
                lambda amounts: amounts[row["index"]] / (10 ** row["decimals"]) if amounts else None
            )
            sub_df["chain_id"] = chain.chain_id
            sub_df["token_address"] = row["token_address"]
            sub_df["destination_vault_address"] = row["destination_vault_address"]

            new_destination_token_values_rows.extend(
                [DestinationTokenValues.from_record(r) for r in sub_df.to_dict(orient="records")]
            )

        destinations_df.apply(lambda row: _extract_destination_token_values(row), axis=1)

        insert_avoid_conflicts(
            new_destination_token_values_rows,
            DestinationTokenValues,
            index_elements=[
                DestinationTokenValues.block,
                DestinationTokenValues.chain_id,
                DestinationTokenValues.token_address,
                DestinationTokenValues.destination_vault_address,
            ],
        )

        # primary source of idle
        idle_destination_token_values = _fetch_idle_destination_token_values(chain, missing_blocks)

        insert_avoid_conflicts(
            idle_destination_token_values,
            DestinationTokenValues,
            index_elements=[
                DestinationTokenValues.block,
                DestinationTokenValues.chain_id,
                DestinationTokenValues.token_address,
                DestinationTokenValues.destination_vault_address,
            ],
        )


def _fetch_idle_destination_token_values(chain: ChainData, missing_blocks: list[int]) -> list[DestinationTokenValues]:

    autopools: list[Autopools] = get_full_table_as_orm(
        Autopools,
        where_clause=Autopools.chain_id == chain.chain_id,
    )

    def _asset_breakdown_to_idle(success, args):
        if success:
            totalIdle, totalDebt, totalDebtMin, totalDebtMax = args
            return int(totalIdle) / 1e18  # maybe 1e6 if autoUSD

    idle_calls = [
        Call(
            autopool.autopool_vault_address,
            ["getAssetBreakdown()((uint256,uint256,uint256,uint256))"],
            [(autopool.autopool_vault_address, _asset_breakdown_to_idle)],
        )
        for autopool in autopools
    ]

    idle_df = get_raw_state_by_blocks(idle_calls, missing_blocks, chain, include_block_number=True)

    idle_destination_token_values = []

    def _extract_idle_destination_token_values(row: dict):
        for autopool_vault_address, total_idle in row.items():
            if autopool_vault_address != "block":
                this_autopool = [a for a in autopools if a.autopool_vault_address == autopool_vault_address][0]
                idle_destination_token_values.append(
                    DestinationTokenValues(
                        block=int(row["block"]),
                        chain_id=chain.chain_id,
                        destination_vault_address=autopool_vault_address,
                        token_address=this_autopool.base_asset,
                        spot_price=1.0,
                        quantity=total_idle,
                    )
                )

    idle_df.apply(_extract_idle_destination_token_values, axis=1)
    return idle_destination_token_values


token_spot_prices_and_reserves_df, destinations_df = ensure_destination_token_values_are_current()

2025-05-02 12:25:39,981 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-02 12:25:39,981 INFO sqlalchemy.engine.Engine SELECT max(blocks.block) AS block 
FROM blocks 
WHERE blocks.chain_id = %(chain_id_1)s AND blocks.block >= %(block_1)s AND blocks.block <= %(block_2)s GROUP BY date_trunc(%(date_trunc_1)s, blocks.datetime) ORDER BY date_trunc(%(date_trunc_2)s, blocks.datetime)
2025-05-02 12:25:39,982 INFO sqlalchemy.engine.Engine [cached since 1146s ago] {'chain_id_1': 1, 'block_1': 20752910, 'block_2': 100000000, 'date_trunc_1': 'day', 'date_trunc_2': 'day'}
2025-05-02 12:25:40,237 INFO sqlalchemy.engine.Engine COMMIT
2025-05-02 12:25:40,325 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-02 12:25:40,326 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destination_token_values
            WHERE destination_token_values.chain_id = 1
        
2025-05-02 12:25:40,326 INFO sqlalchemy.engine.Engine [cached since 1146s ago] {}
2025-05-02 12:25:41,783 INFO sq

In [25]:
row = destinations_df.iloc[0]
row

token_address                0xae78736Cd615f374D3085123A210448E74Fc6393
index                                                                 0
chain_id                                                              1
destination_vault_address    0xf3ae3c74EaD129e770A864CeE291A805b170bBe0
underlying_symbol                                         B-rETH-STABLE
pool                         0x1E19CF2D73a72Ef1332C882F20534B6519Be0276
decimals                                                             18
symbol                                                             rETH
Name: 0, dtype: object

In [62]:
all_destination_token_records = []


def _extract_destination_token_values(row: dict) -> None:
    token_spot_price_column = (row["pool"], row["token_address"], "spot_price")
    quantity_column = (row["destination_vault_address"], "underlyingReserves_amounts")

    sub_df = token_spot_prices_and_reserves_df[["block", quantity_column, token_spot_price_column]].copy()
    sub_df.columns = ["block", "quantity", "spot_price"]
    sub_df["quantity"] = sub_df["quantity"].apply(
        lambda amounts: amounts[row["index"]] / (10 ** row["decimals"]) if amounts else None
    )
    sub_df["chain_id"] = row["chain_id"]
    sub_df["token_address"] = row["token_address"]
    sub_df["destination_vault_address"] = row["destination_vault_address"]

    all_destination_token_records.extend(
        [DestinationTokenValues.from_record(r) for r in sub_df.to_dict(orient="records")]
    )


destinations_df.apply(lambda row: _extract_destination_token_values(row), axis=1)

destinationt_token_values_df = pd.DataFrame.from_records((r.to_record() for r in all_destination_token_records))
destinationt_token_values_df

,block,chain_id,token_address,destination_vault_address,spot_price,quantity
0,20759464,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1.117733,5870.693219
1,20974401,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1.119200,5979.056377
2,21189371,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1.115141,4852.323881
3,21404196,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1.121225,5320.089727
4,21619004,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1.119486,7106.733644
...,...,...,...,...,...,...
1299,21404196,1,0x856c4Efb76C1D1AE02e20CEB03A2A6a08b0b8dC3,0x3772973f8F399D74488D5cF3276C032E0afC8A6f,0.999580,NaN
1300,21619004,1,0x856c4Efb76C1D1AE02e20CEB03A2A6a08b0b8dC3,0x3772973f8F399D74488D5cF3276C032E0afC8A6f,0.999621,NaN
1301,21833845,1,0x856c4Efb76C1D1AE02e20CEB03A2A6a08b0b8dC3,0x3772973f8F399D74488D5cF3276C032E0afC8A6f,0.999709,NaN
1302,22048553,1,0x856c4Efb76C1D1AE02e20CEB03A2A6a08b0b8dC3,0x3772973f8F399D74488D5cF3276C032E0afC8A6f,0.999838,NaN


[DestinationTokenValues(block=20759464, chain_id=1, token_address='0xae78736Cd615f374D3085123A210448E74Fc6393', destination_vault_address='0xf3ae3c74EaD129e770A864CeE291A805b170bBe0', spot_price=1.1177331375912674, quantity=5870.693219026985),
 DestinationTokenValues(block=20974401, chain_id=1, token_address='0xae78736Cd615f374D3085123A210448E74Fc6393', destination_vault_address='0xf3ae3c74EaD129e770A864CeE291A805b170bBe0', spot_price=1.1191999342495509, quantity=5979.05637672614),
 DestinationTokenValues(block=21189371, chain_id=1, token_address='0xae78736Cd615f374D3085123A210448E74Fc6393', destination_vault_address='0xf3ae3c74EaD129e770A864CeE291A805b170bBe0', spot_price=1.115140986447224, quantity=4852.323881004728),
 DestinationTokenValues(block=21404196, chain_id=1, token_address='0xae78736Cd615f374D3085123A210448E74Fc6393', destination_vault_address='0xf3ae3c74EaD129e770A864CeE291A805b170bBe0', spot_price=1.1212248121487185, quantity=5320.089726638753),
 DestinationTokenValues(bl

In [28]:
row

token_address                0xae78736Cd615f374D3085123A210448E74Fc6393
index                                                                 0
chain_id                                                              1
destination_vault_address    0xf3ae3c74EaD129e770A864CeE291A805b170bBe0
underlying_symbol                                         B-rETH-STABLE
pool                         0x1E19CF2D73a72Ef1332C882F20534B6519Be0276
decimals                                                             18
symbol                                                             rETH
Name: 0, dtype: object

mainnet_launch.database.schema.full.DestinationTokenValues

token_address                0xae78736Cd615f374D3085123A210448E74Fc6393
index                                                                 0
destination_vault_address    0xf3ae3c74EaD129e770A864CeE291A805b170bBe0
underlying_symbol                                         B-rETH-STABLE
pool                         0x1E19CF2D73a72Ef1332C882F20534B6519Be0276
decimals                                                             18
symbol                                                             rETH
Name: 0, dtype: object

# summary stats not correct

In [19]:
df[(df["block"] == 21561709) & (df["destination_vault_address"] == "0x40219bBda953ca811d2D0168Dc806a96b84791d9")].T

,676,1600,1963
destination_vault_address,0x40219bBda953ca811d2D0168Dc806a96b84791d9,0x40219bBda953ca811d2D0168Dc806a96b84791d9,0x40219bBda953ca811d2D0168Dc806a96b84791d9
block,21561709,21561709,21561709
chain_id,1,1,1
incentive_apr,0.050965,0.050965,0.050965
fee_apr,0.001706,0.001706,0.001706
base_apr,0.013472,0.013472,0.013472
points_apr,0.0,0.0,0.0
fee_plus_base_apr,NaN,NaN,NaN
total_apr_in,0.061046,0.061046,0.061046
total_apr_out,0.063189,0.063189,0.063189


In [17]:
duplicated_rows = df[df.duplicated(keep=False)]
duplicated_rows

,destination_vault_address,block,chain_id,incentive_apr,fee_apr,base_apr,points_apr,fee_plus_base_apr,total_apr_in,total_apr_out,underlying_token_total_supply,safe_total_supply,price_per_share,price_return,underlying_safe_price,underlying_spot_price,underlying_backing,safe_backing_discount,spot_backing_discount,safe_spot_spread
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,1,0.041235,0.003997,0.005817,0.0,NaN,0.046281,0.046281,13157.451200,12302.611124,1.036523,-0.000645,1.036523,1.036516,1.035855,0.000645,0.000638,0.000007
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20809562,1,0.034270,0.002351,0.005887,0.0,NaN,0.038545,0.038545,13268.592709,12494.240544,1.036546,-0.000536,1.036546,1.036537,1.035992,0.000535,0.000527,0.000009
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20859740,1,0.030393,0.001475,0.006061,0.0,NaN,0.034644,0.034644,13157.904657,12386.417861,1.036394,-0.000245,1.036394,1.036388,1.036140,0.000245,0.000239,0.000006
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20909955,1,0.034249,0.004201,0.006074,0.0,NaN,0.040596,0.040596,13154.734411,12383.247616,1.036953,-0.000504,1.036953,1.036957,1.036431,0.000504,0.000508,-0.000004
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20960089,1,0.034002,0.002755,0.006241,0.0,NaN,0.039572,0.042442,13142.590915,12368.757665,1.036537,-0.000026,1.036537,1.036534,1.036510,0.000026,0.000023,0.000003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2206,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,22163172,1,0.068757,0.000603,0.000000,0.0,NaN,0.062485,0.063346,6354.562864,5864.555184,1.000312,0.000000,1.000312,1.000485,1.001462,-0.001149,-0.000975,-0.000174
2207,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,22213305,1,0.068766,0.000709,0.000000,0.0,NaN,0.062598,0.062741,6007.236756,5560.819667,1.001248,0.000000,1.001248,1.001240,1.001439,-0.000191,-0.000199,0.000008
2208,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,22263499,1,0.068747,0.001677,0.000000,0.0,NaN,0.063550,0.063829,6178.529435,5428.673462,1.001128,0.000000,1.001128,1.001137,1.001501,-0.000373,-0.000363,-0.000009
2209,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,22313650,1,0.070801,0.002718,0.000000,0.0,NaN,0.066439,0.067697,6044.275638,5306.396226,1.000068,0.000000,1.000068,1.000067,1.001750,-0.001678,-0.001680,0.000002


In [7]:
duplicated_rows = df[df.duplicated(keep=False)]

# if you just want one representative per set of duplicates, you can then:
unique_duplicate_keys = duplicated_rows.drop_duplicates()
unique_duplicate_keys

,Unnamed: 0,destination_vault_address,block,chain_id,incentive_apr,fee_apr,base_apr,points_apr,fee_plus_base_apr,total_apr_in,...,underlying_token_total_supply,safe_total_supply,price_per_share,price_return,underlying_safe_price,underlying_spot_price,underlying_backing,safe_backing_discount,spot_backing_discount,safe_spot_spread


In [4]:
import plotly.express as px


df.pivot(index="block", columns="destination_vault_address", values="underlying_token_total_supply")

ValueError: Index contains duplicate entries, cannot reshape

In [1]:
df.columns

NameError: name 'df' is not defined